In [2]:
import pandas as pd
import numpy as np
import gc

train_tx = pd.read_csv("data/ieee-fraud-detection/train_transaction.csv")
train_id = pd.read_csv("data/ieee-fraud-detection/train_identity.csv")
df = train_tx.merge(train_id, on="TransactionID", how="left")

del train_tx
gc.collect()

print(df.shape)

(590540, 434)


In [3]:
df["has_identity"] = df["TransactionID"].isin(train_id["TransactionID"])
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["day"] = df["TransactionDT"] // 86400

C:\Users\Siddhu\AppData\Local\Temp\ipykernel_31096\2382287862.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["has_identity"] = df["TransactionID"].isin(train_id["TransactionID"])
C:\Users\Siddhu\AppData\Local\Temp\ipykernel_31096\2382287862.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hour"] = (df["TransactionDT"] // 3600) % 24
C:\Users\Siddhu\AppData\Local\Temp\ipykernel_31096\2382287862.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

In [4]:
print("Before:", df.memory_usage(deep=True).sum() / 1e9, "GB")

Before: 2.090621739 GB


In [5]:
missing = df.isnull().mean()
cols_to_drop = missing[missing > 0.9].index.tolist()
print(len(cols_to_drop), "columns to drop")
df = df.drop(columns=cols_to_drop)
print(df.shape)

12 columns to drop
(590540, 425)


In [6]:
float_cols = df.select_dtypes("float64").columns
for c in float_cols:
    df[c] = df[c].astype("float32")

In [7]:
obj_cols = df.select_dtypes("object").columns
for c in obj_cols:
    df[c] = df[c].astype("category")

C:\Users\Siddhu\AppData\Local\Temp\ipykernel_31096\647715498.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes("object").columns


In [8]:
print("After:", df.memory_usage(deep=True).sum() / 1e9, "GB")

After: 0.966770036 GB


In [9]:
df.to_parquet("data/clean.parquet")
print("saved")

saved


## Day 3 cleaning decisions
- Dropped N columns that were >90% missing (mostly id_ columns, missing because 76% of transactions have no identity data).
- Did not impute missing values — XGBoost/LightGBM handle NaN natively, and filling now would leak information.
- Downcast float64 to float32 to roughly halve memory.
- Converted text columns to category dtype.
- Added has_identity, hour, and day features for later modeling.